# 03 · Build Clean Manifest

**Purpose.** Turn the raw folders into one clean table: one row per unique, readable image with label and patient.

| | |
|---|---|
| **Reads** | `data/raw/*`, `labels.py` |
| **Writes** | `data/processed/manifest_clean.csv`, `reports/dataset_audit/{cleaning_log, exact_duplicates, near_duplicate_pairs}.csv` |
| **Run time (Colab T4)** | 20-40 min the first time (every image is read once); seconds afterwards (cached) |
| **Next notebook** | 04 · EDA & Data Quality |

Run the cells from top to bottom. All shared logic lives in `src/dermalens/`; this
notebook only calls it and explains each decision.

## 0. Setup

The same cell starts every notebook. It (1) connects Google Drive on Colab,
(2) installs missing libraries on Colab, (3) makes `src/` importable,
(4) finds every project folder and (5) loads `configs/baseline.yaml`.

In [ ]:
import os
import sys
from pathlib import Path

# 1) On Colab, connect Google Drive so the project folder is visible.
try:
    from google.colab import drive                      # this import only works on Colab
    drive.mount("/content/drive")
    os.environ.setdefault("DERMALENS_PROJECT_DIR", "/content/drive/MyDrive/DermaLens_AI")
    IN_COLAB = True
except ImportError:
    IN_COLAB = False                                    # local run: set DERMALENS_PROJECT_DIR yourself

# 2) Where the data lives (PROJECT_DIR) and where the code lives (CODE_DIR, normally the same folder).
PROJECT_DIR = Path(os.environ.get("DERMALENS_PROJECT_DIR", Path.cwd().parent))
CODE_DIR = Path(os.environ.get("DERMALENS_CODE_DIR", PROJECT_DIR))

# 3) Colab starts empty every session: install the few libraries it does not ship with.
if IN_COLAB:
    import subprocess
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", str(CODE_DIR / "requirements.txt")], check=True)

# 4) Make `import dermalens` work.
sys.path.insert(0, str(CODE_DIR / "src"))

from dermalens.config import load_config
from dermalens.paths import ProjectPaths
from dermalens.utils import set_seed

# 5) Folders and settings.
paths = ProjectPaths(PROJECT_DIR).ensure()
CONFIG = load_config(os.environ.get("DERMALENS_CONFIG", CODE_DIR / "configs" / "baseline.yaml"))
set_seed(CONFIG["seed"])
print(paths.describe())

## 1. Build the image table from raw metadata

Each source has its own builder in `src/dermalens/manifest.py`:

* **DermaCon-IN**: `Image_name` in the metadata *is* the file name. Rows marked "quality insufficient" are dropped.
* **SkinDisNet**: `Image_id` + `.jpg` inside `skindisnet2/Preprocessed/<Folder_name>/` (no row-order guessing).
* **MCSI**: rows with `diagnostic == "normal"` become **Healthy**; each image is its own group.

Every builder checks that each file it lists exists, and stops with a clear error if not.

In [ ]:
import pandas as pd
from dermalens import manifest as mf
from dermalens.labels import model_classes

CLASSES = model_classes(open_set=CONFIG["data"]["open_set"])   # 6 targets (+ "Other")
cleaning_log = []                                  # one line per cleaning step, for the report

def log_step(step, table, note=""):
    cleaning_log.append({"step": step, "images": len(table), "patients": table["group_id"].nunique(), "note": note})
    print(f"{step:<40} {len(table):>6,} images  {table['group_id'].nunique():>5,} patients  {note}")

# Raw size of each source before any label mapping, for the funnel.
for name, builder in mf.SOURCE_BUILDERS.items():
    log_step(f"{name}: all usable rows", builder(paths.raw))

manifest = mf.build_manifest(paths.raw, open_set=CONFIG["data"]["open_set"],
                             other_cap_per_diagnosis=CONFIG["data"]["other_cap_per_diagnosis"])
log_step(f"after label mapping ({len(CLASSES)} classes)", manifest)
display(manifest.groupby(["label", "source"]).size().unstack(fill_value=0))

### v3: the "Other" class (open-set recognition)

A model trained only on 6 classes must pick one of them for *any* photo, so a photo of vitiligo or
acne gets a confident wrong answer. We add a 7th output, **Other**, trained on up to
`other_cap_per_diagnosis` photos of each *non-target* diagnosis (vitiligo, acne, urticaria, ...).
Diagnoses close to a target class (any tinea, eczema, dermatitis, psoriasis, scabies) are excluded
so the model is never taught contradictions. In the app, "Other" is shown as **Not identified**.
This technique is called *outlier exposure* (Hendrycks et al., 2019).

In [ ]:
other = manifest[manifest["label"] == "Other"]
print(f"Other: {len(other)} images from {other['source_label'].nunique()} diagnoses")
display(other["source_label"].value_counts().head(25).to_frame("images"))

### v3: patient metadata (age group, sex, body region)

Dermatologists never diagnose from a picture alone: *where* the rash is and *who* has it matter
(scabies between the fingers, tinea in the groin, eczema in children's skin folds). Each row now has
`meta_age`, `meta_sex` and `meta_regions` (11 body regions, pipe-separated), mapped by
`src/dermalens/metadata.py`. Missing values stay missing - the model learns to cope with that.

In [ ]:
coverage = manifest.assign(has_age=manifest["meta_age"].notna(), has_sex=manifest["meta_sex"].notna(),
                           has_region=manifest["meta_regions"].fillna("").ne(""))
display((coverage.groupby("source")[["has_age", "has_sex", "has_region"]].mean() * 100).round(1))

## 2. Exact duplicates (SHA-256)

A SHA-256 fingerprint is computed from each file's bytes; identical files have identical
fingerprints. Only the ~2,600 manifest images are hashed (not all 43,000 raw files), and results are
cached in `reports/dataset_audit/sha256_cache.csv`, so a re-run takes seconds.
If two identical files ever had **different** labels, the function stops: a person must decide.

In [ ]:
manifest = mf.add_file_hashes(manifest, paths.raw, cache_csv=paths.audit / "sha256_cache.csv")
manifest, exact_copies = mf.drop_exact_duplicates(manifest)
exact_copies.to_csv(paths.audit / "exact_duplicates.csv", index=False)
log_step("after removing exact duplicates", manifest, f"{len(exact_copies)} files were in duplicate groups")
display(exact_copies[["sha256", "image_id", "label", "group_id"]].head(20))

## 3. Decode every image once

Opening each image proves it is readable and measures it: width, height, average colour,
brightness, contrast and a **perceptual hash** (a fingerprint of how the picture *looks*).
Results are cached in `image_properties_cache.csv`. Unreadable files are removed.

In [ ]:
manifest = mf.add_image_properties(manifest, paths.raw, cache_csv=paths.audit / "image_properties_cache.csv")
unreadable = manifest[~manifest["readable"].astype(bool)]
unreadable.to_csv(paths.audit / "unreadable_images.csv", index=False)
manifest = manifest[manifest["readable"].astype(bool)].reset_index(drop=True)
log_step("after removing unreadable images", manifest, f"{len(unreadable)} unreadable")
display(unreadable[["image_id", "error"]].head())

## 4. Near duplicates (same photo, re-saved or resized)

Two files can differ in bytes yet show the same photo (resized, re-compressed). Their perceptual
hashes then differ by only a few of 64 bits. We **do not delete** them; we give them the **same
patient group**, so both land in the same split and cannot leak between train and test.
**Review the pairs table** before trusting it: a pair with two different labels deserves a look.

In [ ]:
MAX_BITS = CONFIG["data"]["near_duplicate_bits"]
pairs = mf.find_near_duplicates(manifest, max_distance=MAX_BITS)
pairs.to_csv(paths.audit / "near_duplicate_pairs.csv", index=False)
print(f"Near-duplicate pairs (<= {MAX_BITS} bits apart): {len(pairs)}")
print("Pairs crossing patient groups:", int((pairs["group_a"] != pairs["group_b"]).sum()))
print("Pairs with different labels (check by eye!):", int((pairs["label_a"] != pairs["label_b"]).sum()))
display(pairs.sort_values("distance").head(20))

manifest = mf.merge_groups_of_near_duplicates(manifest, pairs)
largest_group = manifest["group_id"].value_counts().max()
log_step("after merging near-duplicate groups", manifest, f"largest group = {largest_group} images")

## 5. Final checks and save

In [ ]:
assert manifest["image_id"].is_unique
assert manifest["sha256"].is_unique, "exact duplicates remain"
assert manifest["label"].isin(CLASSES).all()

manifest.to_csv(paths.processed / "manifest_clean.csv", index=False)
pd.DataFrame(cleaning_log).to_csv(paths.audit / "cleaning_log.csv", index=False)

print(f"Saved {len(manifest):,} images from {manifest['group_id'].nunique():,} patient groups.")
display(pd.DataFrame(cleaning_log))

**Done.** `manifest_clean.csv` is the only image table later notebooks use. The cleaning log above is
the "data funnel" table for the report: how many images survived each step and why.